# Week 3, day 1 (afternoon) — Extra practice 07 SOLUTIONS: filtering   (L02)

Executed in the lab image against the real `../data/sales.csv`. Every quoted
number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 07 — Filtering. Run this once.
import pandas as pd

sales = pd.read_csv("../data/sales.csv", parse_dates=["OrderDate"])

print("rows:", len(sales))
print("categories:", list(sales["Category"].unique()))
print("ship modes:", list(sales["ShipMode"].unique()))

### Question 1

Technology `68`, Express Air `41`, **both `9`**.

An `and` is always no larger than either half. Nine rows satisfy both,
out of 300.

Worth noticing how small that intersection is before you compute anything
from it — a mean over 9 rows is a very different object from a mean over
300, and it will move if a single order changes.

In [ ]:
tech = sales["Category"] == "Technology"
express = sales["ShipMode"] == "Express Air"
print("Technology:  ", tech.sum())
print("Express Air: ", express.sum())
print("both:        ", (tech & express).sum())

### Question 2

`tech & ~express` -> `59`; `68 - 9` -> `59`. -> they agree.

Two routes to the same number, which is a cheap way to check a mask you
are unsure of. If a compound condition surprises you, decompose it into
counts and see whether the arithmetic works out.

`~` inverts the mask element by element. It is not `not`, which would raise
the same `ValueError` as Q8.

In [ ]:
tech = sales["Category"] == "Technology"
express = sales["ShipMode"] == "Express Air"
by_mask = (tech & ~express).sum()
by_sub = tech.sum() - (tech & express).sum()
print("with & and ~:", by_mask)
print("by subtraction:", by_sub)
print("agree:", by_mask == by_sub)

### Question 3

`79` orders in 2012. -> earliest `2012-01-11`, latest `2012-12-08`.

`.dt.year` works only because `parse_dates` was passed at load time — the
setup cell did it, so the column is a real timestamp rather than the text
from worksheet 11 Q8.

The date range is worth reading: the 2012 rows stop on 8 December, not 31
December. This is a 300-row sample of a larger file, so 'orders in 2012' is
not a complete year and any month-on-month comparison against it would be
misleading in December.

In [ ]:
y2012 = sales[sales["OrderDate"].dt.year == 2012]
print("orders in 2012:", len(y2012))
print("earliest:", y2012["OrderDate"].min().date())
print("latest:  ", y2012["OrderDate"].max().date())

### Question 4

`19` orders from 2011 onwards over 2000, totalling `75966.99`.

Nineteen rows carrying nearly £76,000 — an average around £4,000 against
the file-wide mean order of a few hundred.

That is what a threshold filter does: it selects the tail, so every summary
of the subset is a summary of the tail. Correct, and not a statement about
orders in general.

In [ ]:
mask = (sales["OrderDate"] >= "2011-01-01") & (sales["Sales"] > 2000)
subset = sales[mask]
print("rows:", len(subset))
print("total Sales:", round(subset["Sales"].sum(), 2))

### Question 5

`79` rows -> `Technology 59`, `Furniture 20`.

`isin` on one column and `~isin` on another, combined with `&`. Office
Supplies is absent from the result, which is the check that the negation
worked.

Compare with writing this as four `|` and `!=` conditions — same answer,
far harder to read and far easier to get wrong by one misplaced
parenthesis.

In [ ]:
mask = (sales["ShipMode"].isin(["Regular Air", "Delivery Truck"])
        & ~sales["Category"].isin(["Office Supplies"]))
print("rows:", mask.sum())
print()
print(sales[mask]["Category"].value_counts())

### Question 6

median Sales `240.76999999999998`. -> loss-making-but-large: `46` rows, mean Discount `0.0526`; everything else: `254` rows, mean Discount `0.0468`.

Two things here.

The median prints as `240.76999999999998`, not `240.77`. It is the average
of the two middle values on an even-length column, so it is a computed
float with the usual representation error — the same phenomenon as
worksheet 01 Q3, showing up somewhere you would not expect it.

The discount comparison is the trap. `0.0526` against `0.0468` looks like a
finding: bigger discounts on the loss-making large orders. It is a
difference of about half a percentage point, computed from 46 rows against
254. That is a suggestive number, not evidence — with 46 rows the mean
moves noticeably if two orders change. Report it as something to
investigate, never as a conclusion.

In [ ]:
median_sales = sales["Sales"].median()
odd = (sales["Profit"] < 0) & (sales["Sales"] > median_sales)

print("median Sales:", median_sales)
print("loss-making but large:", odd.sum(), "rows")
print("  mean Discount:", round(sales[odd]["Discount"].mean(), 4))
print("everything else:", (~odd).sum(), "rows")
print("  mean Discount:", round(sales[~odd]["Discount"].mean(), 4))

### Question 7

`big` has `8` rows and gains a `Flag` column; **`"Flag" in sales.columns` is `False`**. -> after `.loc` assignment on the original, `8` rows are flagged.

Writing into the filtered frame changed the filtered frame and left the
original untouched — the same copy-versus-view lesson as worksheet 06 Q2,
in the form that confuses people most.

Note the solution calls `.copy()` explicitly. Without it, assigning into a
slice raises a warning about chained assignment, because pandas cannot tell
whether you meant to modify the original or the subset. `.copy()` says 'I
mean a separate frame'; `.loc[mask, col] = value` says 'I mean the
original'. Say one of them, deliberately.

In [ ]:
big = sales[sales["Sales"] > 5000].copy()
big["Flag"] = True
print("rows in big:", len(big))
print("'Flag' in sales.columns:", "Flag" in sales.columns)
print()

# The supported way: write into the ORIGINAL through .loc.
sales["Flag"] = False
sales.loc[sales["Sales"] > 5000, "Flag"] = True
print("after .loc assignment, flagged rows:", sales["Flag"].sum())

### Question 8

Using `or` -> **raises** `ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().`

Same error as worksheet 07 Q10, from `or` instead of `and` — the whole
family behaves this way. Python's `or`, `and` and `not` each need one
boolean and cannot collapse a 300-element Series to one.

The suggestions in the message remain a red herring: `.any()` and `.all()`
would silence the error and give you a completely wrong filter. What you
want is `|`, which is not mentioned.

The mapping to memorise: `and` -> `&`, `or` -> `|`, `not` -> `~`, and
parentheses around every condition.

In [ ]:
print(sales[(sales["Category"] == "Technology") or (sales["Sales"] > 5000)])